# Linear Regression That Actually Works - Features, Interactions, Diagnostics

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb04_linear_features_diagnostics_instructor.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Fit and interpret linear regression in a pipeline
2. Create interaction and polynomial features responsibly
3. Diagnose underfitting and overfitting by comparing train and validation scores
4. Build a formal model comparison table and visualize results with bar plots
5. Translate coefficients into business meaning (with caveats)
6. Articulate *why* regularization is needed when feature dimensionality grows

---


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: What Drives the Price?

Your baseline model at **HomeValue Analytics** beats the average, but the CEO wants more: *"Can you tell me which features drive prices the most? Why are coastal houses more expensive?"* Stakeholders don't just want predictions — they want explanations.

Linear regression provides both: a prediction and a coefficient for every feature. But the real test is whether adding more features — interactions, polynomial terms — actually improves validation performance, or whether the added complexity backfires. Today's notebook answers that question concretely and sets up the motivation for regularization in the next notebook.

> **Today's focus:** Building an interpretable linear regression for house prices, reading coefficients as business insights, and discovering through a formal comparison why unregularized polynomial features can catastrophically overfit.

---


In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The `Setup complete!` message confirms that all imports loaded without error. This notebook's import list extends the previous one with `PolynomialFeatures` from scikit-learn's preprocessing module — the tool HomeValue's data team will use to automatically generate interaction and squared terms from the original 8 housing features. The same display settings (`display.precision = 4`, `whitegrid` style, `figure.figsize = (10, 6)`) and **RANDOM_SEED = 474** ensure visual and numerical consistency with earlier notebooks.

**Why this matters:** Verifying imports at the top catches environment mismatches early. If you are running in Google Colab, all these packages come pre-installed; on a local machine you may need `pip install scikit-learn`.

---

## 1. Load Data and Create Splits

We continue with the same California Housing dataset used at HomeValue Analytics: **20,640 census tracts**, 8 features (`MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`), and the continuous target `MedHouseVal`. The identical 60/20/20 split with `RANDOM_SEED = 474` guarantees that our partitions match the previous notebook exactly, so any performance difference is attributable to new modeling techniques — not a different data shuffle.

The test set remains locked; all development uses train + validation only.

> 💡 **Gemini Prompt:** "Load the California Housing dataset from scikit-learn as a DataFrame. Separate the target column MedHouseVal from the features. Then split the data into train (60%), validation (20%), and test (20%) sets using a two-step train_test_split with random_state=474. Print the size of each partition."
>
> **After running, verify:**
> - Train set has approximately 12,384 samples (60% of \~20,640)
> - Validation and test sets each have approximately 4,128 samples (20% each)
> - Target variable y contains only MedHouseVal values, not included in X
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
california = fetch_california_housing(as_frame=True)
df = california.frame
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)} (locked)")

**Reading the output:**

The printed summary shows the same partition sizes as the previous notebook: roughly **12,384 training**, **4,128 validation**, and **4,128 test** samples, with the test set marked `(locked)`. Because we use the identical `RANDOM_SEED = 474` and the same two-stage `train_test_split` procedure, every census tract lands in exactly the same partition as before. This reproducibility is essential: if the splits differed, any performance comparison with the \~USD 53,000-MAE linear model from the previous notebook would be meaningless.

**Key takeaway:** Consistent splits across notebooks let HomeValue attribute performance changes entirely to modeling choices (new features, interactions), not to random variation in the data partition.

---

## 2. Baseline Linear Model with Preprocessing

Before adding any feature engineering, we re-fit the same `StandardScaler` + `LinearRegression` pipeline from the previous notebook. This gives HomeValue a clean reference point: the best a plain linear model can do with the original 8 features (`MedInc`, `HouseAge`, `AveRooms`, etc.).

We report both train and validation R² as well as the gap between them. This gap is our first overfitting diagnostic: a large gap means the model is memorizing training patterns that do not generalize to new neighborhoods.

> 💡 **Gemini Prompt:** "Create a scikit-learn Pipeline with two steps: StandardScaler for feature normalization and LinearRegression as the estimator. Fit it on the training data, then compute and print the R-squared score on both the training and validation sets, along with the overfitting gap (train R-squared minus val R-squared)."
>
> **After running, verify:**
> - Both Train R-squared and Val R-squared are in the range 0.59--0.62
> - The overfit gap is very small (close to 0.00), indicating low variance
> - Pipeline has exactly two named steps: scaler and regressor
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Simple linear regression pipeline
baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

baseline_pipeline.fit(X_train, y_train)

train_score = baseline_pipeline.score(X_train, y_train)
val_score = baseline_pipeline.score(X_val, y_val)

print("=== BASELINE LINEAR MODEL ===")
print(f"Train R²: {train_score:.4f}")
print(f"Val R²: {val_score:.4f}")
print(f"Overfit gap: {train_score - val_score:.4f}")

**Reading the output:**

The baseline linear model reports **Train R²** and **Val R²**, both likely in the range of **0.60–0.61**, matching the results from the previous notebook. The "Overfit gap" (train minus validation R²) should be very small (under **0.01**), confirming that a plain 8-feature linear model does not overfit HomeValue's dataset. This is expected: ordinary linear regression with only 8 features relative to \~12,000 training tracts has low variance.

**Why this matters:** This baseline R² is the number to beat. Every feature-engineering step below — adding the `MedInc × AveRooms` interaction, expanding to degree-2 polynomials — must be judged by whether it raises validation R² without dramatically increasing the overfit gap. If a new feature boosts training R² but not validation R², HomeValue is memorizing noise.

---

## 3. Coefficient Interpretation

The CEO at HomeValue doesn't just want predictions — she wants explanations: *"Why is this neighborhood predicted at USD 350k while that one is USD 180k?"* Linear regression answers this question directly: each feature gets a coefficient that quantifies its contribution to the predicted price.

**What standardized coefficients tell HomeValue:**
- **Direction**: A positive coefficient on `MedInc` means wealthier neighborhoods have higher prices
- **Relative importance**: After scaling, the largest coefficient identifies the feature that moves prices the most
- **Magnitude**: A coefficient of 0.85 on `MedInc` means a one-standard-deviation increase in median income raises the predicted price by USD 85,000

**Interpretation caveats the pricing team must remember:**
- Coefficients measure *association*, not *causation* — a large `Latitude` coefficient reflects geographic price patterns, not that moving north changes prices
- Multicollinearity between features (e.g., `AveRooms` and `AveBedrms`) can make individual coefficients unstable
- These interpretations assume the relationship is linear — a strong assumption we will test below

> 💡 **Gemini Prompt:** "Extract the linear regression coefficients from the fitted baseline pipeline, pair each with its feature name, and sort by absolute value. Display the sorted table, then create a horizontal bar chart showing each coefficient. Also print the intercept and a note explaining that positive coefficients increase predictions while negative ones decrease them."
>
> **After running, verify:**
> - MedInc (median income) has the largest absolute coefficient
> - Bar chart has a red dashed vertical line at x=0 separating positive from negative effects
> - Intercept value is printed below the chart (should be close to the mean of y_train)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Extract coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': baseline_pipeline.named_steps['regressor'].coef_,
    'Abs_Coefficient': np.abs(baseline_pipeline.named_steps['regressor'].coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("=== LINEAR REGRESSION COEFFICIENTS ===")
print(coefficients)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(coefficients['Feature'], coefficients['Coefficient'])
plt.xlabel('Coefficient Value (after scaling)')
plt.title('Feature Importance via Linear Regression Coefficients')
plt.axvline(x=0, color='red', linestyle='--')
plt.tight_layout()
plt.show()

print(f"\nIntercept: {baseline_pipeline.named_steps['regressor'].intercept_:.4f}")
print(f"\n💡 Positive coefficients increase the prediction")
print(f"💡 Negative coefficients decrease the prediction")

**Reading the output:**

The table lists all 8 features sorted by absolute coefficient magnitude. Because features were standardized first, the coefficient sizes are directly comparable. **`MedInc`** (median income) almost certainly dominates, confirming what HomeValue's real estate agents already know — neighborhood income is the strongest price driver. The horizontal bar chart makes this hierarchy visual: long bars indicate influential features, and the direction (positive or negative) shows whether the feature pushes predicted `MedHouseVal` up or down. The printed intercept is the model's predicted value when all standardized features equal zero (i.e., at the training-set mean of every feature).

A positive coefficient on `MedInc` and negative coefficients on `Latitude`/`Longitude` likely reflect California's geography: wealthier, coastal, lower-latitude areas (think San Francisco, LA) command higher prices. The pricing team can now weight features *objectively* rather than relying on intuition about which neighborhoods are "hot."

**Key takeaway:** Standardized coefficients are useful for ranking feature importance within HomeValue's model, but remember they measure *association*, not *causation*. A large coefficient on `Latitude` does not mean moving south causes price increases; it reflects geographic price patterns baked into the data.

---

## 4. Feature Engineering: Interactions

### When to Add Interactions

HomeValue's agents know something the raw features miss: the effect of income on price *depends on the neighborhood's housing stock*. A high-`MedInc` tract with large `AveRooms` (spacious suburban homes) behaves differently from a high-`MedInc` tract with small rooms (urban condos). An **interaction term** `MedInc × AveRooms` captures this combined effect that neither variable alone represents.

**Warning:** Interactions increase features — every pair of features creates a new column. With 8 features, all pairwise interactions would add 28 new columns. We start with one hand-picked interaction based on domain knowledge before automating the process.

> 💡 **Gemini Prompt:** "Manually create an interaction feature by multiplying MedInc and AveRooms columns in both the training and validation sets. Fit a new StandardScaler + LinearRegression pipeline on the augmented training data (now 9 features), evaluate on both sets, and print the R-squared improvement over the baseline model."
>
> **After running, verify:**
> - New column MedInc_x_AveRooms appears in both training and validation DataFrames
> - Feature count increased from 8 to 9
> - Improvement over baseline is printed (may be positive or near-zero)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Let's create a simple interaction manually first
# Example: interaction between MedInc and AveRooms
X_train_interact = X_train.copy()
X_val_interact = X_val.copy()

X_train_interact['MedInc_x_AveRooms'] = X_train['MedInc'] * X_train['AveRooms']
X_val_interact['MedInc_x_AveRooms'] = X_val['MedInc'] * X_val['AveRooms']

# Fit model with interaction
interact_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

interact_pipeline.fit(X_train_interact, y_train)

train_score_interact = interact_pipeline.score(X_train_interact, y_train)
val_score_interact = interact_pipeline.score(X_val_interact, y_val)
val_mae_baseline = mean_absolute_error(y_val, baseline_pipeline.predict(X_val))
val_mae_interact = mean_absolute_error(y_val, interact_pipeline.predict(X_val_interact))

print("=== WITH INTERACTION FEATURE ===")
print(f"Train R²: {train_score_interact:.4f}")
print(f"Val R²: {val_score_interact:.4f}")
print(f"\nImprovement over baseline (Val R²): {val_score_interact - val_score:.4f}")

# --- Bar plot: Baseline vs With Interaction ---
step1_df = pd.DataFrame([
    {'Model': 'Baseline Linear', 'Features': X_train.shape[1],
     'Train_R2': train_score, 'Val_R2': val_score, 'Val_MAE': val_mae_baseline},
    {'Model': 'With Interaction', 'Features': X_train_interact.shape[1],
     'Train_R2': train_score_interact, 'Val_R2': val_score_interact, 'Val_MAE': val_mae_interact},
])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, title in zip(axes, ['Train_R2', 'Val_R2', 'Val_MAE'],
                              ['Train R²', 'Validation R²', 'Validation MAE']):
    step1_df.plot.bar(x='Model', y=metric, ax=ax, legend=False, color=['steelblue', 'coral'])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', fontsize=9)
fig.suptitle('Step 1: Does One Manual Interaction Help?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

After adding the hand-crafted `MedInc_x_AveRooms` interaction feature, the model now has **9 features** instead of 8. The printed train and validation R² values show whether this single interaction term improved HomeValue's predictions. The "Improvement over baseline" line reports the difference in validation R² — typically a small positive number (a few thousandths), reflecting the fact that the income-times-rooms product captures a joint pricing effect not fully represented by either `MedInc` or `AveRooms` alone.

For HomeValue, this interaction reveals that high-income tracts with spacious homes are *disproportionately* expensive — an effect that neither income nor room count captures individually. The pricing algorithm should capture these synergies to avoid systematic under-prediction in upscale suburbs.

**Why this matters:** Manually crafting interactions forces you to think about domain knowledge: *which* feature combinations have a plausible combined effect on `MedHouseVal`? This is more interpretable than blindly generating all possible interactions, and it helps the pricing team explain the model's logic to clients.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Add an interaction or polynomial block and measure validation change.

**Instructions:**
1. Use `PolynomialFeatures` to create degree=2 features
2. Fit a new pipeline
3. Compare validation scores
4. Watch for overfitting!

---

> 💡 **Gemini Prompt:** "Build a three-step Pipeline that first applies PolynomialFeatures with degree=2 and include_bias=False, then StandardScaler, then LinearRegression. Fit on the original training data, score on both train and validation, and print the R-squared values, overfit gap, original feature count, expanded feature count, and the feature-explosion multiplier. Then create a 3-panel bar plot comparing Baseline Linear, With Interaction, and Polynomial (deg=2) on Train R², Validation R², and Validation MAE — to make any overfitting visible. **Warning:** the polynomial model may have a catastrophically negative validation R² due to overfitting, so set the Validation R² plot's y-axis limits manually (e.g., [-2, 1]) and add a text annotation showing the actual value when it falls off-chart."
>
> **After running, verify:**
> - Feature count expands from 8 to 44 (originals + squares + pairwise products)
> - Train R² is noticeably higher than baseline (polynomial fits the training data better)
> - **Validation R² is dramatically lower than baseline — possibly strongly negative — revealing severe overfitting**
> - Validation MAE is noticeably higher than baseline (the model is actually worse on new data)
> - Bar plot clearly shows the train vs. validation divergence across all three models
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance
#
# What you need to build:
#   1. A scikit-learn Pipeline: PolynomialFeatures(degree=2, include_bias=False)
#      → StandardScaler → LinearRegression
#   2. Fit it on the original X_train, y_train and compute train / validation R²
#   3. Print the train R², val R², overfit gap, and feature counts
#   4. Compute the feature-explosion multiplier (new feature count / original)
#
# Variables available from earlier cells:
#   - X_train, X_val, y_train, y_val
#   - train_score, val_score (baseline), train_score_interact, val_score_interact
#
# Key question to answer in the YOUR ANALYSIS cell below:
#   Why is the polynomial model's validation R² so different from its training R²?


### YOUR ANALYSIS:

**Observation 1: Performance**  
[Did polynomial features improve validation score?]

**Observation 2: Overfitting**  
[Is the train-val gap larger now?]

**Observation 3: Complexity**  
[Is the added complexity worth the improvement?]

---

## 5. Feature Engineering Reflection and Model Comparison

So far we have explored three distinct feature engineering strategies for HomeValue's pricing model:

| Strategy | Features | Philosophy |
|---|---|---|
| **Baseline Linear** | 8 (original) | Trust the raw measurements — no engineering |
| **With Interaction** | 9 (baseline + MedInc × AveRooms) | Domain knowledge — wealthy neighborhoods with spacious homes command premium prices |
| **Polynomial deg=2** | 44 (all squares + all pairs) | Brute force — let the model decide which combinations matter |

Each strategy represents a different stance on the **bias-variance tradeoff**. The baseline is conservative (high bias, low variance). The interaction adds targeted flexibility. The polynomial cranks flexibility to the maximum — at a price we are about to quantify.

In practice, data scientists almost never know in advance which strategy will win. They build a comparison and let the validation numbers decide. In the next exercise, you will do exactly that — and the result will set up the motivation for the entire next notebook on regularization.

---


## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Build a comprehensive comparison of all three feature engineering strategies, visualize the results, and reflect on what they teach us about the bias-variance tradeoff.

Your goals:

1. **Build the comparison table** — a pandas DataFrame with one row per model (Baseline Linear, With Interaction, Polynomial deg=2) and columns: `Model`, `Features`, `Train_R2`, `Val_R2`, `Overfit_Gap`, `Val_MAE`.
2. **Create a 3-panel bar plot** — Train R², Validation R² (with y-axis clipped to `[-2, 1]` so the polynomial disaster does not hide the other two bars), and Validation MAE. Annotate any off-chart values.
3. **Identify the best model** by validation R² and print it.
4. **Write your interpretation** in the cell below, addressing:
   - Which model had the best *validation* performance, and by how much?
   - Why did the polynomial model — which should in theory capture more signal — collapse on validation data?
   - What does this tell you about the tradeoff between model capacity and generalization?
   - What kind of constraint would make the polynomial feature space deployable for HomeValue?

This is not just a comparison exercise — it is the setup for the next notebook. The polynomial model's failure is the single best motivation for why **regularization** (Ridge and Lasso, the next notebook) is essential.

---


> 💡 **Gemini Prompt:** "Build a pandas DataFrame called `comparison` with exactly three rows — Baseline Linear, With Interaction, and Polynomial (deg=2) — and the columns Model, Features, Train_R2, Val_R2, Overfit_Gap, Val_MAE. Compute each Val_MAE with `mean_absolute_error(y_val, pipeline.predict(X_val))` for the corresponding pipeline. Print the table without the index, then print the model with the best validation R-squared. Finally create a 3-panel bar plot: Train R² (all positive), Validation R² with y-axis limits set to [-2, 1] and red-text annotations for any off-chart values, and Validation MAE (always positive)."
>
> **After running, verify:**
> - Table has exactly 3 rows with all 6 columns populated
> - Feature counts are 8, 9, and 44 respectively
> - The best Val_R² belongs to the Baseline or Interaction model (NOT the polynomial)
> - The polynomial model's Overfit_Gap is enormous (train score high, val score catastrophically low or negative)
> - Bar plot clearly shows Train R² going up across models while Val R² collapses for the polynomial
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance
#
# What you need to build:
#   1. A pandas DataFrame called `comparison` with the three models (Model, Features,
#      Train_R2, Val_R2, Overfit_Gap, Val_MAE)
#   2. A 3-panel bar plot (Train R², Val R² clipped, Val MAE)
#   3. A printout naming the best model by Val_R2
#
# Variables available from earlier cells:
#   - baseline_pipeline, train_score, val_score
#   - interact_pipeline, X_train_interact, X_val_interact, train_score_interact, val_score_interact
#   - poly_pipeline, train_score_poly, val_score_poly, n_features_poly
#
# Hint for the clipped Val R² panel:
#   ax.set_ylim(-2, 1)
#   for i, v in enumerate(comparison['Val_R2']):
#       label = f'{v:.4f}' if -2 <= v <= 1 else f'{v:.2f} ⚠️'
#       ax.text(i, max(min(v, 0.9), -1.9), label, ha='center',
#               fontsize=9, fontweight='bold', color='red' if v < -1 else 'black')


### YOUR INTERPRETATION:

**Question 1 — Which model had the best validation performance, and by how much?**

[Your answer here]

**Question 2 — Why did the polynomial model collapse on validation data despite looking good on training data?**

[Your answer here]

**Question 3 — What does this teach you about the tradeoff between model capacity (more features) and generalization?**

[Your answer here]

**Question 4 — What kind of constraint would make the polynomial feature space deployable for HomeValue?**

[Your answer here]

---


## 6. Why We Need Regularization (Bridge to the next notebook)

This notebook has taught HomeValue's data team a hard lesson: **more features are not free**. Unregularized polynomial features can take a working model and break it, producing predictions so unstable they could never be deployed. The polynomial model's training R² looks great, but its validation R² collapses into negative territory — a textbook signature of severe overfitting driven by correlated high-dimensional features.

The natural question is: *can we keep the richer polynomial feature space while preventing this disaster?* The answer is **yes**, and the technique is called **regularization**.

**Regularization adds a penalty on coefficient magnitudes** to the linear regression objective function. Instead of solving:

$$\min_w \|y - Xw\|^2$$

we solve:

$$\min_w \|y - Xw\|^2 + \alpha \cdot R(w)$$

where $R(w)$ is a penalty term that grows with the size of the coefficients, and $\alpha$ controls how strongly the penalty is enforced. Two penalty forms dominate in practice:

- **Ridge regression** uses $R(w) = \sum_i w_i^2$ (L2 penalty). It shrinks all coefficients toward zero proportionally, stabilizing the model against multicollinearity.
- **Lasso regression** uses $R(w) = \sum_i |w_i|$ (L1 penalty). It shrinks some coefficients to exactly zero, performing automatic feature selection.

Both techniques are the subject of **the next notebook: Regularization (Ridge/Lasso) + Project Proposal**. In that notebook, you will apply Ridge and Lasso to the same California Housing dataset — including the same polynomial features — and watch the model go from catastrophic failure to solid performance. You will see exactly how the penalty term tames the unstable coefficients.

The key insight: **regularization does not throw away the polynomial features — it uses them responsibly**. After the next notebook, HomeValue will be able to use richer feature spaces safely.

---


## 7. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Coefficient Interpretation**: Requires scaling. Watch out for multicollinearity — features that carry overlapping information make individual coefficients unstable.
2. **Feature Engineering (manual)**: One targeted interaction (MedInc × AveRooms) barely moved the validation needle — minimal gain, but grounded in domain knowledge.
3. **The Polynomial Trap**: Expanding to 44 polynomial features *broke* the model. Training R² went up, but validation R² collapsed into negative territory. More features ≠ better generalization.
4. **Validation Discipline**: Train scores can lie. Validation scores tell the truth about generalization. Always cross-check both.
5. **Why Regularization Matters**: this notebook's polynomial failure is the precise problem Ridge and Lasso solve in the next notebook. By penalizing large coefficient magnitudes, regularization lets us keep rich feature spaces *without* the instability.

### Critical Rules:

> **"Scale features before interpreting coefficients"**

> **"More features are not free — unregularized polynomial features can catastrophically overfit"**

> **"Train R² can lie; validation scores and model comparisons tell the truth"**

### Next Steps:

- Next notebook: **Regularization (Ridge/Lasso) + Project proposal**
- Ridge and Lasso will solve today's polynomial failure by constraining coefficient magnitudes
- Start thinking about your project dataset and target

---


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Linear Regression chapter
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Linear methods
- scikit-learn User Guide: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- scikit-learn User Guide: [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

---



<center>

Thank you!

</center>